# Does SW add pairwise information beyond width/resource? — CPU LOSO

Train pair-distance regressions on five trajectory states and evaluate the sixth. Compare Resource against Resource+SW, then convert each held-out prediction into a fixed-uniform-marginal pair policy and evaluate exact variance on the held-out Gram. No GPU, network training, accuracy, or test data is used.

In [ ]:
import os, subprocess, sys, json, time, zipfile
from pathlib import Path
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
os.chdir(PROJECT_ROOT); sys.path.insert(0, str(PROJECT_ROOT))
GIT_COMMIT = subprocess.run(['git','rev-parse','HEAD'],capture_output=True,text=True,check=True).stdout.strip()
print('Commit:', GIT_COMMIT)

## Resolve dynamic-geometry trajectory output and optional HT FLOPs fallback

In [ ]:
import importlib
import rq2_pairwise_incremental_value, rq2_pairwise_surrogate_regret, rq2_quick_trajectory_diagnostic
rq2_pairwise_incremental_value = importlib.reload(rq2_pairwise_incremental_value)
rq2_pairwise_surrogate_regret = importlib.reload(rq2_pairwise_surrogate_regret)
rq2_quick_trajectory_diagnostic = importlib.reload(rq2_quick_trajectory_diagnostic)
QUICK_ROOT = rq2_pairwise_surrogate_regret.find_quick_trajectory_root(
    Path('/kaggle/input'), '/kaggle/working/materialized-sw-incremental-quick'
)
try:
    HT_ROOT = rq2_quick_trajectory_diagnostic.find_ht_development_root(
        Path('/kaggle/input'), '/kaggle/working/materialized-sw-incremental-ht'
    )
except FileNotFoundError:
    HT_ROOT = None
print('Quick trajectory root:', QUICK_ROOT)
print('Optional HT root:', HT_ROOT)

## Run six leave-one-state-out folds and exact held-out policy evaluation

In [ ]:
OUTPUT_DIR = Path('/kaggle/working/rq2-pairwise-sw-incremental-value')
started = time.perf_counter()
metadata = rq2_pairwise_incremental_value.run_pairwise_incremental_value(
    QUICK_ROOT, OUTPUT_DIR, ht_root=HT_ROOT
)
metadata['git_commit'] = GIT_COMMIT
(OUTPUT_DIR/'metadata.json').write_text(json.dumps(metadata, indent=2)+'\n')
print(f"Completed in {time.perf_counter()-started:.1f} seconds")
print(json.dumps(metadata, indent=2))

## Inspect held-out prediction and policy results

In [ ]:
import pandas as pd
from IPython.display import display, Image
for name in ('pairwise_loso_summary.csv','pairwise_loso_metrics.csv','pairwise_loso_policy_variance.csv','pairwise_loso_coefficients.csv'):
    print('\n', name); display(pd.read_csv(OUTPUT_DIR/name))
display(Image(filename=str(OUTPUT_DIR/'loso_pair_distance_mae.png')))
display(Image(filename=str(OUTPUT_DIR/'loso_hybrid_policy_variance_delta.png')))
display(Image(filename=str(OUTPUT_DIR/'loso_hybrid_predicted_vs_observed.png')))

## Validate and export

In [ ]:
required = [
    'pairwise_state_design_matrix.csv','pairwise_loso_predictions.csv',
    'pairwise_loso_metrics.csv','pairwise_loso_coefficients.csv',
    'pairwise_loso_policy_variance.csv','pairwise_loso_summary.csv',
    'loso_pair_distance_mae.png','loso_hybrid_policy_variance_delta.png',
    'loso_hybrid_predicted_vs_observed.png','metadata.json',
]
missing = [name for name in required if not (OUTPUT_DIR/name).is_file() or (OUTPUT_DIR/name).stat().st_size == 0]
assert not missing, f'Missing SW incremental-value artifacts: {missing}'
saved = json.loads((OUTPUT_DIR/'metadata.json').read_text())
assert saved['preprocessing_fit_train_states_only'] is True
assert saved['training_performed'] is False and saved['test_used'] is False
bundle_path = Path('/kaggle/working/rq2-pairwise-sw-incremental-value.zip')
with zipfile.ZipFile(bundle_path, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in OUTPUT_DIR.rglob('*'):
        if path.is_file(): bundle.write(path, path.relative_to(OUTPUT_DIR))
print('Download/persist:', bundle_path, f'{bundle_path.stat().st_size/2**20:.1f} MiB')
bundle_path